# RQ4 Test 2: Question 7 forecast test

Sepehr Mansouri, MSc Applied Computing, BCIT. Supervisor: Dr. Michal Aibin.

**Question 7:** If F1 is added to a model that already has weather, terrain and fire history, and the model is tested on years it has not seen, how much does the score improve?

**Requested next step:** Run the test with leave-one-year-out validation over 2019 to 2025. Report the change in PR-AUC and ROC AUC with an interval taken over fire years. Write the decision rule down with a date before running it. If the greenness measures add no clear gain, RQ5 becomes the main contribution of SP-B.

## Overview

- **Decision rule:** written and sent to the supervisor on 21 October 2026, & agreed on 9 November 2026. Sections 1 to 3 were run on 11 November 2026. Section 4 was added on 23 November 2026 at the supervisor's request.
- **F1:** the April and May greenness of each cell compared with its earlier years only.
- **Models:** Model A (baseline) uses spring weather, snow, terrain, land cover, fire history of the surrounding area and the square. Model B uses the same inputs plus F1. Both are gradient boosting models with fixed settings. A logistic regression is run as a check.
- **Validation:** leave-one-year-out. Each test year is left out in turn, both models are trained on the other years and scored on the year left out.
- **Scores:** PR-AUC (main score) and ROC AUC. The change is Model B minus Model A, averaged over the test years with a 95% t-interval.
- **Clear gain:** the lower end of the PR-AUC interval is above 0 and the mean ROC AUC change is above 0.
- **Backup in the rule:** if F1 exists for less than half of the cells in any test year, the Level-2A version of F1 and the test years 2021 to 2025 are used.

**Input files:** the tables saved by the Test 1 notebook (`MyDrive/rq4_outputs`) and three Earth Engine exports from `rq4_test2_layers.js` (`rq4_l1c_`, `rq4_snow_` and `rq4_static_<square>.tif` in `MyDrive/rq4_s2`).

The results are summarised at the end of the notebook.

## 1. Setup

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


Libraries, settings and four functions:
- `read_named` reads a GeoTIFF and checks its grid.
- `trailing_z` compares each year with the earlier years only.
- `both_mean` averages April and May when both exist.
- `cell_table` turns the 100 × 100 maps of a square into a table with one row per cell.

In [2]:
from pathlib import Path
from datetime import datetime
import warnings
import numpy as np
import pandas as pd
import rasterio
from scipy import stats
from scipy.ndimage import uniform_filter
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
warnings.filterwarnings("ignore", category=RuntimeWarning)

# ---- Folders ----
BASE_DIR   = Path("/content/drive/MyDrive")
RASTER_DIR = BASE_DIR / "rq4_s2"          # Earth Engine exports
OUT_DIR    = BASE_DIR / "rq4_outputs"     # tables saved by the Test 1 notebook
FU_DIR     = OUT_DIR / "followup"         # all new tables go here
FU_DIR.mkdir(parents=True, exist_ok=True)

# ---- The seven study squares (corners in EPSG:3005 metres) ----
SQUARES = pd.DataFrame(
    [("Prince_George_1", 1040000, 1095000, 1140000, 1195000),
     ("Prince_George_2", 1212000, 1221000, 1312000, 1321000),
     ("Kamloops_1",      1362000,  546000, 1462000,  646000),
     ("Kamloops_2",      1387000,  662000, 1487000,  762000),
     ("Cariboo_1",       1095000,  770000, 1195000,  870000),
     ("Southeast_1",     1633000,  595000, 1733000,  695000),
     ("Northwest_1",      878000,  965000,  978000, 1065000)],
    columns=["square_id", "xmin", "ymin", "xmax", "ymax"]).set_index("square_id")

ANALYSIS_YEARS = list(range(2019, 2026))
EXCLUDE = ["burned_before", "early_nbac", "early_firms", "late_nbac", "unknown_nbac", "prescribed_nbac"]
MIN_CLEAR = 2                                   # clear views needed for a valid month
keys = ["square_id", "year", "row", "col"]      # one row = one cell in one year
ROWS, COLS = np.meshgrid(np.arange(100), np.arange(100), indexing="ij")   # row and column of every cell


def read_named(path, sq):
    """Read a GeoTIFF as {band name: 100 x 100 array}, after checking it sits on the square's 1 km grid."""
    out = {}
    with rasterio.open(path) as r:
        t = r.transform
        if r.crs.to_epsg() != 3005 or (r.width, r.height) != (100, 100) or \
           not np.isclose(t.c, SQUARES.at[sq, "xmin"]) or not np.isclose(t.f, SQUARES.at[sq, "ymax"]):
            raise ValueError(f"{path.name}: grid does not match the squares table")
        for b, name in enumerate(r.descriptions, start=1):
            a = r.read(b).astype(np.float32)
            if r.nodata is not None:
                a[a == r.nodata] = np.nan
            out[name] = a
    return out


def trailing_z(x, valid, min_prev):
    """Compare each year with EARLIER years only. x, valid: arrays (years, 100, 100).
    Returns z = (value - mean of earlier years) / their standard deviation, and diff = value - mean."""
    z = np.full(x.shape, np.nan, np.float32)
    d = np.full(x.shape, np.nan, np.float32)
    for i in range(1, x.shape[0]):                       # the first year has no earlier years
        prev = np.where(valid[:i], x[:i], np.nan)
        n = np.isfinite(prev).sum(axis=0)
        with np.errstate(invalid="ignore", divide="ignore"):
            mu, sd = np.nanmean(prev, axis=0), np.nanstd(prev, axis=0, ddof=1)
            ok = valid[i] & (n >= min_prev)
            d[i] = np.where(ok, x[i] - mu, np.nan)
            z[i] = np.where(ok & (sd > 0), (x[i] - mu) / sd, np.nan)
    return z, d


def both_mean(a, b):
    """Average of two months, only where both have a value."""
    return np.where(np.isfinite(a) & np.isfinite(b), (a + b) / 2.0, np.nan)


def cell_table(sq, year=None, **layers):
    """Turn 100 x 100 layers of one square into a table with one row per cell."""
    head = {"square_id": sq} if year is None else {"square_id": sq, "year": year}
    return pd.DataFrame({**head, "row": ROWS.ravel(), "col": COLS.ravel(),
                         **{name: a.ravel() for name, a in layers.items()}})


print("Section A ready")

Section A ready


## 2. Inputs

The Test 1 table is rebuilt with the same removals. The counts should match Test 1 (490,000, 361,200 and 4,740).

In [3]:
# ---- Rebuild the Test 1 table with the same removals ----
feat = pd.read_parquet(OUT_DIR / "rq4_cell_year_features.parquet")
refs = pd.read_parquet(OUT_DIR / "rq4_cell_year_reference.parquet")
outc = pd.read_parquet(OUT_DIR / "rq4_cell_year_outcomes.parquet")
for t in (feat, refs, outc):
    t["square_id"] = t["square_id"].astype(str)

cm = pd.read_parquet(OUT_DIR / "rq4_cell_month.parquet",
                     columns=["square_id", "year", "month", "row", "col", "gcc", "n_clear", "snow_frac",
                              "z_gcc", "diff_gcc", "veg_ok"])
cm["square_id"] = cm["square_id"].astype(str)
spring_cm = cm[cm.month.isin([4, 5])]                     # April and May only

# F1_raw (Test 1 version): mean April and May difference, both months needed
spring = spring_cm.pivot_table(index=keys, columns="month", values="diff_gcc")
f1_raw = spring.mean(axis=1).where(spring.notna().all(axis=1)).rename("F1_raw").reset_index()

joined = (feat.merge(refs, on=keys, how="left").merge(f1_raw, on=keys, how="left")
              .merge(outc, on=keys, how="inner"))
joined.loc[~joined.veg_ok, "F1_raw"] = np.nan
joined = joined[joined.year.isin(ANALYSIS_YEARS) & joined.nbac_year_available].copy()
joined["burned"] = joined["burned"].astype(bool)
joined["square_year"] = joined.square_id + "_" + joined.year.astype(str)
joined["keep"] = joined.veg_ok & ~joined[EXCLUDE].any(axis=1)     # same removals as Test 1
ana = joined[joined.keep].copy()
print(f"{len(joined):,} cell-years before masks, {len(ana):,} kept, {int(ana.burned.sum()):,} burned among kept")
print("(the main notebook reported 490,000, 361,200 and 4,740)")

490,000 cell-years before masks, 361,200 kept, 4,740 burned among kept
(the main notebook reported 490,000, 361,200 and 4,740)


F1 from Level-2A, earlier years only. Level-2A starts in 2019 and each cell needs 2 earlier years. This version therefore starts in 2021.

In [4]:
# ---- F1 from Level-2A, earlier years only (at least 2 earlier valid years) ----
years_cm = sorted(cm.year.unique())
iy = {y: i for i, y in enumerate(years_cm)}
frames = []
for sq in SQUARES.index:
    z, diff = {}, {}
    for m in (4, 5):
        c = spring_cm[(spring_cm.square_id == sq) & (spring_cm.month == m)]
        i = c.year.map(iy).values
        x = np.full((len(years_cm), 100, 100), np.nan, np.float32)     # greenness, one map per year
        v = np.zeros(x.shape, bool)                                     # valid month (2+ clear views)
        x[i, c.row.values, c.col.values] = c.gcc.values
        v[i, c.row.values, c.col.values] = (c.n_clear.values >= MIN_CLEAR) & np.isfinite(c.gcc.values)
        z[m], diff[m] = trailing_z(x, v, 2)
    F1, F1_raw = both_mean(z[4], z[5]), both_mean(diff[4], diff[5])
    frames += [cell_table(sq, y, F1_trail_l2a=F1[iy[y]], F1_trail_l2a_raw=F1_raw[iy[y]]) for y in years_cm]
trail_l2a = pd.concat(frames, ignore_index=True)
print("F1_trail_l2a cells with a value, by year:")
print(trail_l2a.groupby("year").F1_trail_l2a.count())

F1_trail_l2a cells with a value, by year:
year
2019        0
2020        0
2021    56754
2022    51213
2023    63929
2024    56327
2025    63120
2026    66134
Name: F1_trail_l2a, dtype: int64


F1 from Level-1C, earlier years only. Level-1C goes back to 2015. The table shows the share of cells with at least 2 clear views in each month. 2016 and 2017 have fewer clear images.

In [5]:
# ---- F1 from Level-1C, earlier years only (the main version in the rule) ----
L1C_YEARS = list(range(2016, 2026))
frames, cover = [], []
for sq in SQUARES.index:
    b = read_named(RASTER_DIR / f"rq4_l1c_{sq}.tif", sq)
    z, diff = {}, {}
    for m in (4, 5):
        x = np.stack([b[f"gcc_{y}_{m:02d}"] for y in L1C_YEARS])
        n = np.stack([b[f"n_clear_{y}_{m:02d}"] for y in L1C_YEARS])
        v = np.isfinite(x) & (n >= MIN_CLEAR)
        z[m], diff[m] = trailing_z(x, v, 2)
        cover += [{"square_id": sq, "year": y, "month": m, "share_valid": float(v[k].mean())}
                  for k, y in enumerate(L1C_YEARS)]
    F1, F1_raw = both_mean(z[4], z[5]), both_mean(diff[4], diff[5])
    frames += [cell_table(sq, y, F1_trail_l1c=F1[k], F1_trail_l1c_raw=F1_raw[k]) for k, y in enumerate(L1C_YEARS)]
trail_l1c = pd.concat(frames, ignore_index=True)

# Share of cells with a valid month, by year (used for the backup check in section 3)
cov = pd.DataFrame(cover).pivot_table(index="year", columns="month", values="share_valid")
cov.to_csv(FU_DIR / "g3c_l1c_coverage.csv")
print("Level-1C: mean share of cells with at least 2 clear observations"); display(cov.round(2))

Level-1C: mean share of cells with at least 2 clear observations


month,4,5
year,,
2016,0.23,0.61
2017,0.10,0.30
2018,0.95,0.97
2019,0.95,0.98
2020,0.99,0.92
2021,1.00,0.91
2022,0.95,0.75
2023,0.92,1.00
2024,0.97,0.81


Both F1 versions are added to the table.

In [6]:
ana2 = ana.merge(trail_l2a, on=keys, how="left").merge(trail_l1c, on=keys, how="left")
ana2.to_parquet(FU_DIR / "followup_analysis_table.parquet", index=False)
print("Joined:", len(ana2), "cell-years")

Joined: 361200 cell-years


Snow from ERA5-Land: April snow water equivalent and May snow cover.

In [7]:
# ---- Snow: April snow water equivalent and May snow cover (ERA5-Land) ----
frames = []
for sq in SQUARES.index:
    b = read_named(RASTER_DIR / f"rq4_snow_{sq}.tif", sq)
    frames += [cell_table(sq, y, swe_apr=b[f"swe_{y}_04"], snowc_may=b[f"snowc_{y}_05"]) for y in range(2016, 2026)]
ana3 = ana2.merge(pd.concat(frames, ignore_index=True), on=keys, how="left")

Land cover shares (ESA WorldCover 2021) and terrain (CDEM), one value per cell.

In [8]:
# ---- Land cover shares and terrain: one value per cell, same in every year ----
STATIC = ["tree", "shrub", "grass", "wetland", "elev", "slope", "northness", "eastness"]
static = pd.concat([cell_table(sq, **{k: read_named(RASTER_DIR / f"rq4_static_{sq}.tif", sq)[k] for k in STATIC})
                    for sq in SQUARES.index], ignore_index=True)
ana4 = ana3.merge(static, on=["square_id", "row", "col"], how="left")
cls = ana4[["tree", "shrub", "grass", "wetland"]]
ana4["cover"] = cls.idxmax(axis=1).where(cls.notna().all(axis=1))     # dominant land cover of the cell

## 3. Question 7 test

Settings copied from the decision rule of 21 October 2026.

In [9]:
# ---- Settings copied from the decision rule ----
Q7_PREREG_DATE = "2026-10-21"                           # date the Q7 rule was sent to the supervisor
TEST_YEARS = list(range(2019, 2026))
PRIMARY_F1 = "F1_trail_l1c"                             # main version: Level-1C from 2016
FALLBACK_F1, FALLBACK_YEARS = "F1_trail_l2a", list(range(2021, 2026))
MIN_F1_SHARE = 0.5                                      # backup if primary F1 covers < 50% of a test year
HGB_PARAMS = dict(learning_rate=0.05, max_iter=300, max_leaf_nodes=31, min_samples_leaf=200,
                  l2_regularization=1.0, early_stopping=False, random_state=20261014)
print("Q7 rule dated", Q7_PREREG_DATE)

Q7 rule dated 2026-10-21


Model A inputs: March to May rain and temperature, May soil moisture, April snow water, May snow cover, terrain, land cover, the share of the surrounding 11 × 11 km that burned before, and the square. NDVI is not one of the inputs of Model A or Model B.

The backup check from the rule follows. F1 from Level-1C exists for only 14% of the 2019 cells, which triggers the backup.

In [10]:
d = ana4.copy()

# ---- Weather: March to May rain and temperature, May soil moisture (ERA5-Land) ----
frames = []
for sq in SQUARES.index:
    b = read_named(RASTER_DIR / f"rq4_era5_{sq}.tif", sq)
    for y in TEST_YEARS:
        frames.append(cell_table(sq, y,
                                 precip_mam=sum(b[f"precip_{y}_{m:02d}"] for m in (3, 4, 5)),
                                 t2m_mam=sum(b[f"t2m_{y}_{m:02d}"] for m in (3, 4, 5)) / 3,
                                 swvl1_may=b[f"swvl1_{y}_05"]))
d = d.merge(pd.concat(frames, ignore_index=True), on=keys, how="left")

# ---- Fire history: share of the 11 x 11 cells around each cell that burned before year y ----
frames = []
fby = outc.drop_duplicates(["square_id", "row", "col"])[["square_id", "row", "col", "first_burn_year"]]
inside = uniform_filter(np.ones((100, 100)), size=11, mode="constant")   # corrects the share at the square edges
for sq in SQUARES.index:
    g = fby[fby.square_id == sq]
    first = np.full((100, 100), np.nan)
    first[g.row.values, g.col.values] = g.first_burn_year.values
    for y in TEST_YEARS:
        burned_before = (np.nan_to_num(first, nan=9999) < y).astype(float)
        frames.append(cell_table(sq, y, burned_before_5km=uniform_filter(burned_before, size=11, mode="constant") / inside))
d = d.merge(pd.concat(frames, ignore_index=True), on=keys, how="left")

# ---- Model A inputs (Model B adds F1) ----
BASE = ["precip_mam", "t2m_mam", "swvl1_may", "swe_apr", "snowc_may",
        "elev", "slope", "northness", "eastness", "tree", "shrub", "grass", "wetland",
        "burned_before_5km", "sq_code"]
d["sq_code"] = pd.Categorical(d.square_id, categories=list(SQUARES.index))

# ---- Backup check from the rule ----
share_f1 = d[d.year.isin(TEST_YEARS)].groupby("year")[PRIMARY_F1].apply(lambda s: s.notna().mean())
print("Share of kept cell-years with the primary F1, by year:"); print(share_f1.round(3))
if (share_f1 < MIN_F1_SHARE).any():
    print("Pre-registered fallback applies:", FALLBACK_F1, FALLBACK_YEARS)
    F1COL, YEARS = FALLBACK_F1, FALLBACK_YEARS
else:
    F1COL, YEARS = PRIMARY_F1, TEST_YEARS
d = d[d.year.isin(YEARS)].reset_index(drop=True)
print(f"Using {F1COL}, test years {YEARS[0]} to {YEARS[-1]}: {len(d):,} cell-years, {int(d.burned.sum()):,} burned")

Share of kept cell-years with the primary F1, by year:
year
2019    0.140
2020    0.866
2021    0.898
2022    0.756
2023    0.918
2024    0.827
2025    0.913
Name: F1_trail_l1c, dtype: float64
Pre-registered fallback applies: F1_trail_l2a [2021, 2022, 2023, 2024, 2025]
Using F1_trail_l2a, test years 2021 to 2025: 253,884 cell-years, 4,609 burned


Leave-one-year-out. For each test year, Model A and Model B are trained on the other years and scored on the test year.

In [11]:
def fit_predict_hgb(cols, tr, te):
    """Main model: gradient boosting with the fixed settings from the rule."""
    m = HistGradientBoostingClassifier(**HGB_PARAMS, categorical_features="from_dtype")
    m.fit(tr[cols], tr.burned.astype(int))
    return m.predict_proba(te[cols])[:, 1]


def fit_predict_logit(cols, tr, te):
    """Check model: logistic regression. Inputs are scaled and missing values filled using the training years only."""
    num = [c for c in cols if c != "sq_code"]

    def prep(x, ref):
        out = {}
        for c in num:
            v = x[c].clip(-5, 5) if c.startswith("F1") else x[c]       # limit extreme F1 values
            has = ref[c].notna().any()
            sd = ref[c].std()
            sd = sd if np.isfinite(sd) and sd > 0 else 1.0
            out[c] = ((v.fillna(ref[c].median() if has else 0.0) - (ref[c].mean() if has else 0.0)) / sd).values
            out[c + "_na"] = x[c].isna().astype(float).values          # flag for a missing value
        squares = pd.get_dummies(x.sq_code, drop_first=True).astype(float).reset_index(drop=True)
        return pd.concat([pd.DataFrame(out), squares], axis=1)

    m = LogisticRegression(C=1.0, max_iter=2000)
    m.fit(prep(tr, tr), tr.burned.astype(int))
    return m.predict_proba(prep(te, tr))[:, 1]


# ---- Leave-one-year-out: train on the other years, score on the year left out ----
rows = []
for y in YEARS:
    tr, te = d[d.year != y], d[d.year == y].reset_index(drop=True)
    for model, fn in [("gradient boosting (primary)", fit_predict_hgb), ("logistic (secondary)", fit_predict_logit)]:
        p0 = fn(BASE, tr, te)                 # Model A
        p1 = fn(BASE + [F1COL], tr, te)       # Model B
        rows.append({"model": model, "test_year": y, "n": len(te), "n_burned": int(te.burned.sum()),
                     "roc_base": roc_auc_score(te.burned, p0), "roc_f1": roc_auc_score(te.burned, p1),
                     "pr_base": average_precision_score(te.burned, p0), "pr_f1": average_precision_score(te.burned, p1)})
    print("done", y)
q7 = pd.DataFrame(rows)
q7["d_roc"] = q7.roc_f1 - q7.roc_base        # change from adding F1
q7["d_pr"] = q7.pr_f1 - q7.pr_base
q7.to_csv(FU_DIR / "g6_q7_per_year.csv", index=False)
q7.round(4)

done 2021
done 2022
done 2023
done 2024
done 2025


,model,test_year,n,n_burned,roc_base,roc_f1,pr_base,pr_f1,d_roc,d_pr
0,gradient boosting (primary),2021,53361,1767,0.5524,0.5482,0.0371,0.0380,-0.0042,0.0009
1,logistic (secondary),2021,53361,1767,0.7134,0.7180,0.0602,0.0620,0.0045,0.0019
2,gradient boosting (primary),2022,51824,400,0.3399,0.3478,0.0054,0.0055,0.0078,0.0001
3,logistic (secondary),2022,51824,400,0.2913,0.2992,0.0051,0.0055,0.0079,0.0003
4,gradient boosting (primary),2023,50548,1703,0.5588,0.5611,0.0352,0.0365,0.0023,0.0014
5,logistic (secondary),2023,50548,1703,0.5204,0.5226,0.0412,0.0413,0.0023,0.0001
6,gradient boosting (primary),2024,49379,490,0.6595,0.6795,0.0275,0.0397,0.0200,0.0122
7,logistic (secondary),2024,49379,490,0.3526,0.3466,0.0069,0.0069,-0.0059,0.0000
8,gradient boosting (primary),2025,48772,249,0.5144,0.5242,0.0047,0.0049,0.0098,0.0001
9,logistic (secondary),2025,48772,249,0.1014,0.0939,0.0027,0.0027,-0.0075,-0.0000


Average change over the test years with a 95% interval, followed by the decision.

In [12]:
def t_interval(x):
    """Mean and 95% t-interval over the test years."""
    x = np.asarray(x, float); n = len(x)
    h = stats.t.ppf(0.975, n - 1) * x.std(ddof=1) / np.sqrt(n)
    return x.mean(), x.mean() - h, x.mean() + h

rows = []
for model, g in q7.groupby("model"):
    for metric in ["d_pr", "d_roc"]:
        m, lo, hi = t_interval(g[metric])
        rows.append({"model": model, "metric": metric, "mean_change": m, "lo95": lo, "hi95": hi,
                     "years_improved": int((g[metric] > 0).sum()), "years": len(g)})
q7s = pd.DataFrame(rows); q7s.to_csv(FU_DIR / "g6_q7_summary.csv", index=False)
display(q7s.round(4))

# ---- The rule: clear gain = lower end of the PR-AUC interval above 0 AND mean ROC AUC change above 0 ----
prim = q7s[q7s.model.str.startswith("gradient")].set_index("metric")
clear_gain = (prim.loc["d_pr", "lo95"] > 0) and (prim.loc["d_roc", "mean_change"] > 0)
text = (f"Q7 rule dated {Q7_PREREG_DATE}. Feature {F1COL}, test years {YEARS[0]} to {YEARS[-1]}.\n"
        f"Primary model: mean change in PR-AUC {prim.loc['d_pr','mean_change']:+.4f} "
        f"[{prim.loc['d_pr','lo95']:+.4f}, {prim.loc['d_pr','hi95']:+.4f}], "
        f"mean change in ROC AUC {prim.loc['d_roc','mean_change']:+.4f} "
        f"[{prim.loc['d_roc','lo95']:+.4f}, {prim.loc['d_roc','hi95']:+.4f}].\n"
        + ("Result under the rule: CLEAR GAIN. RQ4 stays the main contribution of SP-B."
           if clear_gain else
           "Result under the rule: NO CLEAR GAIN. RQ5 becomes the main contribution of SP-B and RQ4 a secondary finding."))
print(text)
with open(FU_DIR / "g6_q7_readout.txt", "w") as fh:
    fh.write(f"Run {datetime.now():%Y-%m-%d %H:%M}\n" + text + "\n\n" + q7.round(4).to_string() + "\n\n" + q7s.round(4).to_string())

,model,metric,mean_change,lo95,hi95,years_improved,years
0,gradient boosting (primary),d_pr,0.0029,-0.0035,0.0094,5,5
1,gradient boosting (primary),d_roc,0.0072,-0.0040,0.0183,4,5
2,logistic (secondary),d_pr,0.0005,-0.0006,0.0015,4,5
3,logistic (secondary),d_roc,0.0002,-0.0081,0.0086,3,5


Q7 rule dated 2026-10-21. Feature F1_trail_l2a, test years 2021 to 2025.
Primary model: mean change in PR-AUC +0.0029 [-0.0035, +0.0094], mean change in ROC AUC +0.0072 [-0.0040, +0.0183].
Result under the rule: NO CLEAR GAIN. RQ5 becomes the main contribution of SP-B and RQ4 a secondary finding.


## 4. Baseline checks (requested 19 November 2026)

Two checks on Model A, the baseline, with the same data and settings as section 3. They do not change the decision above.

1. Cells, burned cells and burn rate for each test year, with the baseline PR-AUC divided by the burn rate. A model that ranks cells at random has a PR-AUC about equal to the burn rate, which means a ratio near 1 indicates no skill.
2. The ROC AUC within each square. The ROC AUC in section 3 ranks all cells of all seven squares together. It can be low because the model gives high values to the wrong squares, even if it ranks cells well inside each square. The AUC within each square removes the differences between squares. Test year 2022 (ROC AUC 0.34) is the main case. The other test years are shown for comparison.

In [13]:
# ---- Burn rate per test year, and baseline PR-AUC as a multiple of it (about 1 = random) ----
yr = q7[q7.model.str.startswith("gradient")].copy()
yr["burn_rate"] = yr.n_burned / yr.n
yr["pr_base_over_burn_rate"] = yr.pr_base / yr.burn_rate
tab_rate = yr[["test_year", "n", "n_burned", "burn_rate", "pr_base", "pr_base_over_burn_rate", "roc_base"]].rename(
    columns={"n": "cells", "n_burned": "burned", "pr_base": "pr_auc_A", "pr_base_over_burn_rate": "pr_auc_A_over_burn_rate",
             "roc_base": "roc_auc_A"})
tab_rate.to_csv(FU_DIR / "g7_burn_rate_by_year.csv", index=False)
tab_rate.round(4)

,test_year,cells,burned,burn_rate,pr_auc_A,pr_auc_A_over_burn_rate,roc_auc_A
0,2021,53361,1767,0.0331,0.0371,1.1208,0.5524
2,2022,51824,400,0.0077,0.0054,0.6983,0.3399
4,2023,50548,1703,0.0337,0.0352,1.0436,0.5588
6,2024,49379,490,0.0099,0.0275,2.7720,0.6595
8,2025,48772,249,0.0051,0.0047,0.9243,0.5144


Where the 2022 fires were: cells, burned cells and burn rate per square among the 2022 test cells, with the number of NBAC fires in the fire season and their area inside each square (from the Test 1 table `rq4_region_year_outcomes.csv`, before the removals).

In [14]:
# ---- Where the 2022 fires were, by square ----
CHECK_YEAR = 2022
ry_out = pd.read_csv(OUT_DIR / "rq4_region_year_outcomes.csv").astype({"square_id": str})
fires = ry_out[ry_out.year == CHECK_YEAR].set_index("square_id")[["n_outcome_fires", "outcome_ha_in_square"]]
fires_sq = (d[d.year == CHECK_YEAR].groupby("square_id").agg(cells=("burned", "size"), burned=("burned", "sum"))
            .join(fires))
fires_sq["burn_rate"] = fires_sq.burned / fires_sq.cells
fires_sq["share_of_burned"] = fires_sq.burned / fires_sq.burned.sum()
fires_sq = fires_sq.sort_values("burned", ascending=False)
fires_sq.to_csv(FU_DIR / f"g7_fires_by_square_{CHECK_YEAR}.csv")
print(f"{CHECK_YEAR}: {int(fires_sq.burned.sum()):,} burned test cells in {int((fires_sq.burned > 0).sum())} of 7 squares")
fires_sq.round(4)

2022: 400 burned test cells in 7 of 7 squares


,cells,burned,n_outcome_fires,outcome_ha_in_square,burn_rate,share_of_burned
square_id,,,,,,
Prince_George_2,8235,273,7,19569.6635,0.0332,0.6825
Southeast_1,6720,67,12,2843.2877,0.0100,0.1675
Kamloops_2,8164,23,12,709.3416,0.0028,0.0575
Kamloops_1,7721,18,15,86.5186,0.0023,0.0450
Cariboo_1,4783,9,3,227.4259,0.0019,0.0225
Prince_George_1,9137,9,5,76.3122,0.0010,0.0225
Northwest_1,7064,1,1,3.6901,0.0001,0.0025


Model A is refitted for each test year with the same inputs, settings and random seed as section 3, and the overall ROC AUC is checked against section 3. The prediction for every cell is saved. For each square with at least one burned and one unburned cell, the table gives the ROC AUC within that square, the burn rate and the mean predicted value.

In [15]:
# ---- Refit Model A for each test year (same split, settings and seed as section 3) ----
check_rows, sq_rows, preds = [], [], []
for y in YEARS:
    tr, te = d[d.year != y], d[d.year == y].reset_index(drop=True)
    te = te.assign(p_base=fit_predict_hgb(BASE, tr, te))
    roc_refit = roc_auc_score(te.burned, te.p_base)
    roc_s3 = yr.loc[yr.test_year == y, "roc_base"].iloc[0]                # should be identical
    check_rows.append({"test_year": y, "roc_auc_A_section3": roc_s3, "roc_auc_A_refit": roc_refit,
                       "difference": roc_refit - roc_s3})
    preds.append(te[keys + ["burned", "p_base"]])
    # ROC AUC inside each square (needs at least one burned and one unburned cell)
    for sq, g in te.groupby("square_id"):
        nb = int(g.burned.sum())
        sq_rows.append({"test_year": y, "square_id": sq, "cells": len(g), "burned": nb, "burn_rate": nb / len(g),
                        "mean_pred_A": g.p_base.mean(),
                        "roc_auc_within": roc_auc_score(g.burned, g.p_base) if 0 < nb < len(g) else np.nan})
    print("done", y)

pd.concat(preds, ignore_index=True).to_parquet(FU_DIR / "g7_baseline_predictions.parquet", index=False)
refit_check = pd.DataFrame(check_rows)
within = pd.DataFrame(sq_rows)
within.to_csv(FU_DIR / "g7_baseline_auc_within_square.csv", index=False)
print("Refit check, largest difference from section 3:", f"{refit_check.difference.abs().max():.2e}")
display(refit_check.round(4))

print(f"\n{CHECK_YEAR}, Model A by square (squares with no burned cell have no AUC):")
display(within[within.test_year == CHECK_YEAR].sort_values("burned", ascending=False).round(4))

print("\nROC AUC within each square, all test years:")
within.pivot_table(index="square_id", columns="test_year", values="roc_auc_within").round(3)

done 2021
done 2022
done 2023
done 2024
done 2025
Refit check, largest difference from section 3: 0.00e+00


,test_year,roc_auc_A_section3,roc_auc_A_refit,difference
0,2021,0.5524,0.5524,0.0
1,2022,0.3399,0.3399,0.0
2,2023,0.5588,0.5588,0.0
3,2024,0.6595,0.6595,0.0
4,2025,0.5144,0.5144,0.0



2022, Model A by square (squares with no burned cell have no AUC):


,test_year,square_id,cells,burned,burn_rate,mean_pred_A,roc_auc_within
12,2022,Prince_George_2,8235,273,0.0332,0.0006,0.5708
13,2022,Southeast_1,6720,67,0.0100,0.0111,0.4903
9,2022,Kamloops_2,8164,23,0.0028,0.0110,0.5534
8,2022,Kamloops_1,7721,18,0.0023,0.0046,0.5611
7,2022,Cariboo_1,4783,9,0.0019,0.0209,0.6087
11,2022,Prince_George_1,9137,9,0.0010,0.0033,0.5732
10,2022,Northwest_1,7064,1,0.0001,0.0064,0.2279



ROC AUC within each square, all test years:


test_year,2021,2022,2023,2024,2025
square_id,,,,,
Cariboo_1,0.939,0.609,0.805,0.589,0.857
Kamloops_1,0.626,0.561,0.540,0.600,0.625
Kamloops_2,0.287,0.553,0.356,0.658,0.281
Northwest_1,0.669,0.228,0.744,0.542,0.907
Prince_George_1,0.453,0.573,0.597,0.421,0.079
Prince_George_2,NaN,0.571,0.590,0.568,0.538
Southeast_1,0.456,0.490,0.565,0.658,0.358


## Summary of results

### Test 1 (17 October 2026): link between spring greenness and fire

F1 compared with all other years of the same cell. Fire years 2019 to 2025, 361,200 cell-years, 4,740 burned. AUC = chance that a random burned cell has a higher value than a random unburned cell (0.5 means no link).

| Measure | AUC | 95% interval |
|---|---|---|
| F1, spring greenness anomaly | 0.717 | 0.598 to 0.804 |
| F3, greenness in August and September of the year before | 0.817 | 0.732 to 0.885 |
| MODIS NDVI, spring (comparison) | 0.731 | 0.634 to 0.803 |

F1 stayed significant with NDVI and weather in the logistic regression (p < 0.001). Comparing whole squares year by year showed no link (Spearman −0.08, p = 0.60). Test 1 shows a link, not a forecast, and F1 also used years after each fire.

### Test 2 (11 November 2026): Question 7 forecast test

F1 compared with earlier years only. Backup applied: Level-2A F1, test years 2021 to 2025, 253,884 cell-years, 4,609 burned.

| Test year | Cells | Burned | Burn rate | PR-AUC A | PR-AUC B | Change | ROC AUC A | ROC AUC B | Change |
|---|---|---|---|---|---|---|---|---|---|
| 2021 | 53,361 | 1,767 | 3.31% | 0.0371 | 0.0380 | +0.0009 | 0.5524 | 0.5482 | −0.0042 |
| 2022 | 51,824 | 400 | 0.77% | 0.0054 | 0.0055 | +0.0001 | 0.3399 | 0.3478 | +0.0078 |
| 2023 | 50,548 | 1,703 | 3.37% | 0.0352 | 0.0365 | +0.0014 | 0.5588 | 0.5611 | +0.0023 |
| 2024 | 49,379 | 490 | 0.99% | 0.0275 | 0.0397 | +0.0122 | 0.6595 | 0.6795 | +0.0200 |
| 2025 | 48,772 | 249 | 0.51% | 0.0047 | 0.0049 | +0.0001 | 0.5144 | 0.5242 | +0.0098 |

| Model | PR-AUC change (95% interval) | ROC AUC change (95% interval) |
|---|---|---|
| Gradient boosting (main) | +0.0029 (−0.0035 to +0.0094) | +0.0072 (−0.0040 to +0.0183) |
| Logistic regression (check) | +0.0005 (−0.0006 to +0.0015) | +0.0002 (−0.0081 to +0.0086) |

**Decision:** the lower end of the PR-AUC interval is below 0. Adding F1 gives no clear gain under the rule. RQ5 becomes the main contribution of SP-B and RQ4 a secondary finding.

**Limits:** the backup reduced the test from 7 to 5 fire years. The baseline model had little skill on unseen years (PR-AUC close to the share of burned cells, ROC AUC below 0.5 in 2022). The comparisons with NDVI and all 13 Sentinel-2 bands are not included.